In [1]:
from pathlib import Path

# ---- USER INPUT ----
# gunw_path = Path("/path/to/NISAR_L2_GUNW_file.h5")
# OR if you want a folder:
gunw_dir = Path("/Volumes/Fortress_L3/SnowWaterEquivalent/NISAR_Samples")
pattern = "*.h5"

wsdlurl = "https://hydroportal.cuahsi.org/Snotel/cuahsi_1_1.asmx?WSDL"

obs_hour = 0                 # 0 = midnight UTC, 6 = 6 AM UTC etc.
reference_date = "12-01"
include_temperature = True

# NISAR footprint defaults
frequency = "A"
pol = "HH"


In [2]:
from utils.nisar_utils import (
    nisar_dates_from_gunw_h5,
    nisar_union_footprints,
)

gunw_files = sorted(gunw_dir.glob(pattern))
print("n GUNW files:", len(gunw_files))

# unique sorted acquisition dates across all files
dates = sorted({d for f in gunw_files for d in nisar_dates_from_gunw_h5(f)})

# union footprint across all files
footprint = nisar_union_footprints(
    gunw_files,
    frequency=frequency,
    pol=pol,
    crs_out="EPSG:4326",
)

print("n dates:", len(dates), "| first/last:", dates[0], dates[-1])
footprint


CPLE_AppDefined in DeprecationWarning: 'Memory' driver is deprecated since GDAL 3.11. Use 'MEM' onwards. Further messages of this type will be suppressed.


n GUNW files: 2
n dates: 4 | first/last: 2008-10-12 00:00:00 2025-01-18 00:00:00


,geometry
0,"MULTIPOLYGON (((-117.95305 34.16261, -117.9534..."


In [3]:
from utils.snotel_utils import (
    fetch_snotel_sites,
    filter_sites_by_polygon,
    fetch_snotel_timeseries,
)

# 1) Fetch all SNOTEL sites (EPSG:4326 points)
sites = fetch_snotel_sites(wsdlurl)

# 2) Filter to footprint geometry (assumes footprint is EPSG:4326)
snotel_sites = filter_sites_by_polygon(sites, footprint.iloc[0].geometry)
print("Stations in footprint:", len(snotel_sites))

# 3) Fetch SWE + optional temperature for those sites
swe_data = fetch_snotel_timeseries(
    snotel_sites,
    wsdlurl,
    start_date=str(dates[0].date()),
    end_date=str(dates[-1].date()),
    reference_date=reference_date,
    obs_hour=obs_hour,
    include_temperature=include_temperature,
)

print("Stations with data returned:", len(swe_data))

# Quick peek
first_station = next(iter(swe_data))
first_station, swe_data[first_station].head()


Stations in footprint: 0
Stations with data returned: 0


StopIteration: 

In [5]:
from utils.plotting import make_footprint_station_map

m = make_footprint_station_map(footprint, snotel_sites, zoom_start=8)
m


In [ ]:
from utils import plot_snotel_data

plot_snotel_data(swe_data, dates, x_axis="days_since_reference")


In [ ]:
from utils import save_pickle

save_pickle(swe_data, "cache/snotel_data_nisar.pkl")
